# BioBERT Mechanism Analyzer — CompoundIQ
**Owner:** Oluchi  
**Project:** CompoundIQ / PharmAI  
**Course:** ITAI 2376 Deep Learning  

---
## What This Does
BioBERT is the **entry point** of the CompoundIQ pipeline. It takes a natural-language mechanism description (e.g. "reduce pain via COX-2 inhibition") and outputs:
1. A **768-dim embedding** that captures the pharmacological meaning
2. **Identified drug targets** (enzymes, receptors, channels)
3. A **confidence score** for each target
4. **Mechanism classification** (what type of drug action this describes)

This feeds directly into the JT-VAE for molecular generation.

In [ ]:
!pip install transformers torch pandas numpy scikit-learn -q

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import re
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded")

## Step 1: Load BioBERT and Build Target Knowledge Base

In [ ]:
class BioBERTMechanismAnalyzer:
    """
    Enhanced BioBERT component for CompoundIQ pipeline.
    Goes beyond raw embeddings — identifies specific drug targets,
    classifies mechanism types, and provides confidence scoring.
    """

    # Known pharmacological targets with keywords and categories
    TARGET_DATABASE = {
        # Enzymes
        'COX-1': {'keywords': ['cox-1', 'cyclooxygenase-1', 'prostaglandin'], 'category': 'enzyme', 'description': 'Cyclooxygenase-1 enzyme'},
        'COX-2': {'keywords': ['cox-2', 'cyclooxygenase-2', 'prostaglandin', 'inflammation'], 'category': 'enzyme', 'description': 'Cyclooxygenase-2 enzyme'},
        'HMG-CoA Reductase': {'keywords': ['hmg-coa', 'statin', 'cholesterol synthesis', 'mevalonate'], 'category': 'enzyme', 'description': 'HMG-CoA reductase (cholesterol synthesis)'},
        'ACE': {'keywords': ['ace', 'angiotensin-converting', 'angiotensin converting', 'blood pressure', 'renin'], 'category': 'enzyme', 'description': 'Angiotensin-converting enzyme'},
        'CYP2C19': {'keywords': ['cyp2c19', 'cytochrome', 'cyp enzyme'], 'category': 'enzyme', 'description': 'Cytochrome P450 2C19'},
        'CYP3A4': {'keywords': ['cyp3a4', 'cytochrome 3a4', 'drug metabolism'], 'category': 'enzyme', 'description': 'Cytochrome P450 3A4'},
        'DHFR': {'keywords': ['dhfr', 'dihydrofolate', 'folate synthesis', 'folate'], 'category': 'enzyme', 'description': 'Dihydrofolate reductase'},
        'Thrombin': {'keywords': ['thrombin', 'coagulation', 'clotting factor', 'anticoagulant'], 'category': 'enzyme', 'description': 'Thrombin (coagulation cascade)'},
        'PDE5': {'keywords': ['pde5', 'phosphodiesterase', 'cgmp', 'vasodilation'], 'category': 'enzyme', 'description': 'Phosphodiesterase type 5'},

        # Receptors
        'GABA-A': {'keywords': ['gaba', 'gaba-a', 'gabaa', 'gamma-aminobutyric', 'sedation', 'anxiolytic'], 'category': 'receptor', 'description': 'GABA-A receptor'},
        'Opioid Receptors': {'keywords': ['opioid', 'mu receptor', 'morphine', 'endorphin', 'mu-opioid'], 'category': 'receptor', 'description': 'Opioid receptors (mu, kappa, delta)'},
        'Histamine H1': {'keywords': ['histamine', 'h1 receptor', 'antihistamine', 'allergy'], 'category': 'receptor', 'description': 'Histamine H1 receptor'},
        'Histamine H2': {'keywords': ['h2 receptor', 'h2 blocker', 'stomach acid', 'gastric'], 'category': 'receptor', 'description': 'Histamine H2 receptor'},
        'Dopamine D2': {'keywords': ['dopamine', 'd2 receptor', 'antipsychotic', 'dopaminergic'], 'category': 'receptor', 'description': 'Dopamine D2 receptor'},
        'Serotonin 5-HT': {'keywords': ['serotonin', '5-ht', 'ssri', 'reuptake'], 'category': 'receptor', 'description': 'Serotonin receptors'},
        'Beta-Adrenergic': {'keywords': ['beta blocker', 'beta-adrenergic', 'adrenergic', 'beta receptor', 'heart rate'], 'category': 'receptor', 'description': 'Beta-adrenergic receptors'},
        'Estrogen Receptor': {'keywords': ['estrogen', 'estradiol', 'hormonal', 'endocrine'], 'category': 'receptor', 'description': 'Estrogen receptors (ER-alpha, ER-beta)'},

        # Ion Channels
        'L-type Calcium Channels': {'keywords': ['calcium channel', 'l-type', 'calcium blocking', 'amlodipine'], 'category': 'ion_channel', 'description': 'L-type calcium channels'},
        'Sodium Channels': {'keywords': ['sodium channel', 'nav', 'local anesthetic', 'voltage-gated sodium'], 'category': 'ion_channel', 'description': 'Voltage-gated sodium channels'},
        'Potassium Channels': {'keywords': ['potassium channel', 'herg', 'k+ channel'], 'category': 'ion_channel', 'description': 'Potassium channels'},

        # Other Targets
        'Heme Polymerization': {'keywords': ['heme', 'hemozoin', 'plasmodium', 'malaria', 'chloroquine'], 'category': 'parasite_target', 'description': 'Heme polymerization (Plasmodium)'},
        'Cell Wall Synthesis': {'keywords': ['cell wall', 'beta-lactam', 'penicillin', 'peptidoglycan', 'bacterial'], 'category': 'bacterial_target', 'description': 'Bacterial cell wall synthesis'},
        'Ribosome (30S)': {'keywords': ['30s ribosome', 'tetracycline', 'aminoglycoside', 'protein synthesis'], 'category': 'bacterial_target', 'description': 'Bacterial 30S ribosomal subunit'},
        'Ribosome (50S)': {'keywords': ['50s ribosome', 'macrolide', 'azithromycin', 'erythromycin'], 'category': 'bacterial_target', 'description': 'Bacterial 50S ribosomal subunit'},
        'DNA Gyrase': {'keywords': ['dna gyrase', 'topoisomerase', 'fluoroquinolone', 'ciprofloxacin'], 'category': 'bacterial_target', 'description': 'DNA gyrase / topoisomerase IV'},
        'Tubulin': {'keywords': ['tubulin', 'microtubule', 'mitotic', 'taxane', 'vincristine'], 'category': 'cancer_target', 'description': 'Tubulin / microtubules'},
    }

    # Mechanism action types
    MECHANISM_TYPES = {
        'inhibitor': ['inhibit', 'block', 'antagoni', 'suppress', 'reduce', 'decrease', 'prevent', 'anti-'],
        'agonist': ['agonist', 'activate', 'stimulat', 'enhance', 'increase', 'promote', 'potentiat'],
        'modulator': ['modulat', 'regulat', 'adjust', 'balance', 'stabiliz'],
        'blocker': ['block', 'channel block', 'receptor block'],
    }

    def __init__(self):
        print("Loading BioBERT (dmis-lab/biobert-v1.1)...")
        self.tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")
        self.model = AutoModel.from_pretrained("dmis-lab/biobert-v1.1")
        self.model.eval()

        # Pre-compute embeddings for all target keywords
        self._build_target_embeddings()
        print(f"BioBERT READY — {len(self.TARGET_DATABASE)} targets loaded")

    def _build_target_embeddings(self):
        """Pre-compute BioBERT embeddings for each target's keywords."""
        self.target_embeddings = {}
        with torch.no_grad():
            for target_name, info in self.TARGET_DATABASE.items():
                keyword_text = f"{target_name}: {info['description']}. Keywords: {', '.join(info['keywords'])}"
                inputs = self.tokenizer(keyword_text, return_tensors="pt", max_length=128, truncation=True, padding=True)
                outputs = self.model(**inputs)
                emb = outputs.last_hidden_state.mean(dim=1).squeeze()
                self.target_embeddings[target_name] = emb

    def embed(self, text):
        """Generate a 768-dim BioBERT embedding from mechanism text."""
        with torch.no_grad():
            inputs = self.tokenizer(text, return_tensors="pt", max_length=256, truncation=True, padding=True)
            outputs = self.model(**inputs)
            embedding = outputs.last_hidden_state.mean(dim=1)
        return embedding

    def identify_targets(self, text, top_k=5):
        """
        Identify the most likely drug targets from a mechanism description.
        Uses both keyword matching AND semantic similarity via BioBERT.
        """
        text_lower = text.lower()
        query_emb = self.embed(text).squeeze()

        results = []
        for target_name, info in self.TARGET_DATABASE.items():
            # Keyword score: how many keywords appear in the text
            keyword_hits = sum(1 for kw in info['keywords'] if kw in text_lower)
            keyword_score = keyword_hits / len(info['keywords'])

            # Semantic score: cosine similarity between text and target embedding
            target_emb = self.target_embeddings[target_name]
            semantic_score = F.cosine_similarity(query_emb.unsqueeze(0), target_emb.unsqueeze(0)).item()

            # Combined score: weighted blend (keywords are strong signal when present)
            if keyword_score > 0:
                combined = 0.6 * keyword_score + 0.4 * semantic_score
            else:
                combined = 0.3 * semantic_score  # Lower weight for pure semantic

            results.append({
                'target': target_name,
                'category': info['category'],
                'description': info['description'],
                'keyword_score': round(keyword_score, 3),
                'semantic_score': round(semantic_score, 3),
                'confidence': round(combined, 3),
            })

        results.sort(key=lambda x: x['confidence'], reverse=True)
        return results[:top_k]

    def classify_mechanism(self, text):
        """Classify the type of mechanism action."""
        text_lower = text.lower()
        scores = {}
        for mech_type, keywords in self.MECHANISM_TYPES.items():
            hits = sum(1 for kw in keywords if kw in text_lower)
            scores[mech_type] = hits
        if max(scores.values()) == 0:
            return 'unknown'
        return max(scores, key=scores.get)

    def analyze(self, text):
        """
        Full analysis: embedding + targets + mechanism type + confidence.
        This is the main function called by the CompoundIQ pipeline.
        """
        embedding = self.embed(text)
        targets = self.identify_targets(text)
        mechanism_type = self.classify_mechanism(text)

        # Overall confidence: based on top target match
        top_confidence = targets[0]['confidence'] if targets else 0.0

        return {
            'input_text': text,
            'embedding': embedding,
            'embedding_shape': tuple(embedding.shape),
            'identified_targets': targets,
            'mechanism_type': mechanism_type,
            'overall_confidence': round(top_confidence, 3),
            'n_targets_found': sum(1 for t in targets if t['confidence'] > 0.3),
        }

# Initialize
analyzer = BioBERTMechanismAnalyzer()

## Step 2: Test with Mechanism Descriptions

In [ ]:
# Test cases covering the CompoundIQ presentation scenarios
test_mechanisms = [
    "Reduce pain and inflammation via COX-2 selective inhibition with minimal stomach damage",
    "Block HMG-CoA reductase to lower cholesterol",
    "Antagonize histamine H1 receptors for allergy relief",
    "Reduce muscle spasms through calcium channel blocking and GABA enhancement without excessive sedation",
    "Make pain go away",  # Vague input test
    "Treat malaria by inhibiting heme polymerization and blocking folate synthesis simultaneously",
    "Treat bacterial infection by disrupting cell wall synthesis and protein synthesis simultaneously",
    "Lower blood pressure using ACE inhibition",
    "Maximize blood thinning effect by combining multiple anticoagulant mechanisms",
]

print("=" * 90)
print("  BioBERT MECHANISM ANALYSIS — COMPOUNDIQ")
print("=" * 90)

for text in test_mechanisms:
    result = analyzer.analyze(text)
    print(f"\nQUERY: \"{text}\"")
    print(f"  Embedding shape: {result['embedding_shape']}")
    print(f"  Mechanism type:  {result['mechanism_type']}")
    print(f"  Confidence:      {result['overall_confidence']}")
    print(f"  Targets found:   {result['n_targets_found']}")
    print(f"  Top targets:")
    for t in result['identified_targets'][:3]:
        bar = '█' * int(t['confidence'] * 30)
        print(f"    {t['target']:25s} [{t['category']:15s}] conf={t['confidence']:.3f}  {bar}")
    print("-" * 90)

## Step 3: Embedding Similarity Visualization

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# Generate embeddings for all test cases
labels = []
embeddings = []
for text in test_mechanisms:
    emb = analyzer.embed(text).squeeze().detach().numpy()
    embeddings.append(emb)
    labels.append(text[:50] + "..." if len(text) > 50 else text)

emb_matrix = np.array(embeddings)

# Cosine similarity heatmap
sim_matrix = cosine_similarity(emb_matrix)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap
short_labels = [t[:30] + "..." if len(t) > 30 else t for t in test_mechanisms]
im = axes[0].imshow(sim_matrix, cmap='RdYlGn', vmin=0.7, vmax=1.0)
axes[0].set_xticks(range(len(short_labels)))
axes[0].set_yticks(range(len(short_labels)))
axes[0].set_xticklabels([f"Q{i+1}" for i in range(len(short_labels))], fontsize=9)
axes[0].set_yticklabels(short_labels, fontsize=7)
axes[0].set_title("BioBERT Embedding Similarity (Cosine)", fontsize=13, fontweight='bold')
plt.colorbar(im, ax=axes[0], shrink=0.8)

# PCA 2D projection
pca = PCA(n_components=2)
coords = pca.fit_transform(emb_matrix)
axes[1].scatter(coords[:, 0], coords[:, 1], s=120, c=range(len(coords)), cmap='tab10', edgecolors='black', linewidth=0.5)
for i, label in enumerate(short_labels):
    axes[1].annotate(f"Q{i+1}", (coords[i, 0]+0.01, coords[i, 1]+0.01), fontsize=8)
axes[1].set_title("PCA Projection of Mechanism Embeddings", fontsize=13, fontweight='bold')
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('biobert_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nLegend:")
for i, text in enumerate(test_mechanisms):
    print(f"  Q{i+1}: {text}")

## Step 4: Integration Output Format

The `analyzer.analyze()` function returns a dictionary that plugs directly into the CompoundIQ pipeline:

```python
{
    'embedding': tensor([768-dim]),        # → feeds into JT-VAE latent mapping
    'identified_targets': [...],            # → guides molecular candidate filtering
    'mechanism_type': 'inhibitor',          # → constrains GNN scoring
    'overall_confidence': 0.85,             # → passed to safety scoring
}
```

In [ ]:
# Show the full output structure for pipeline integration
demo = analyzer.analyze("Reduce pain and inflammation via COX-2 selective inhibition")

print("PIPELINE OUTPUT STRUCTURE:")
print(f"  embedding shape:     {demo['embedding'].shape}")
print(f"  mechanism_type:      {demo['mechanism_type']}")
print(f"  overall_confidence:  {demo['overall_confidence']}")
print(f"  n_targets_found:     {demo['n_targets_found']}")
print(f"\n  identified_targets:")
for t in demo['identified_targets']:
    print(f"    - {t['target']} ({t['category']}): confidence={t['confidence']}")

print(f"\n  embedding vector (first 10 dims):")
print(f"    {demo['embedding'].squeeze()[:10].tolist()}")
print(f"\nThis output is passed to JT-VAE and Three-Way GNN in the full pipeline.")